<div>

  <b>Escuela Politécnica Nacional</b><br>
  <small>Facultad de Ingeniería de Sistemas</small><br>
  <small>Ingeniería en Ciencias de la Computación</small>

  <hr>

  <div style="display:flex; justify-content:space-between;">
    <div>
      Estudiante: <b>Mateo Cumbal</b><br>
      Fecha: <b>2026-07-04</b>
    </div>
    <div style="text-align:right;">
      Asignatura: <b>Recuperación de la Información</b><br>
      Paralelo: <b>GR1CC</b>
    </div>
  </div>

  <hr>

  <div style="text-align:center;">
    <h1><b>Ejercicio 8 — Bases de Datos Vectoriales</b></h1>
  </div>

</div>

## Objetivo de la práctica

Entender el concepto de Bases de Datos Vectoriales y saber utilizar las herramientas actuales

## Parte 0: Carga del Corpus

Vamos a utilizar la API de Kaggle para acceder al dataset _Wikipedia Text Corpus for NLP and LLM Projects_

El corpus está disponible desde este [link](https://www.kaggle.com/datasets/gzdekzlkaya/wikipedia-text-corpus-for-nlp-and-llm-projects?utm_source=chatgpt.com)

### Actividad

1. Carga el corpus

In [1]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

In [2]:
# Set the path to the file you'd like to load
file_path = "wikipedia_text_corpus.csv"

# Load the latest version
df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "gzdekzlkaya/wikipedia-text-corpus-for-nlp-and-llm-projects",
  file_path,
)

df.head()

Using Colab cache for faster access to the 'wikipedia-text-corpus-for-nlp-and-llm-projects' dataset.


,Unnamed: 0,text
0,1,Anovo\n\nAnovo (formerly A Novo) is a computer...
1,2,Battery indicator\n\nA battery indicator (also...
2,3,"Bob Pease\n\nRobert Allen Pease (August 22, 19..."
3,4,CAVNET\n\nCAVNET was a secure military forum w...
4,5,CLidar\n\nThe CLidar is a scientific instrumen...


## Parte 1: Generación de Embeddings

Vamos a utilizar E5 como modelo de embeddings.

La documentación de E5 está disponible desde este [link](https://huggingface.co/intfloat/e5-base-v2)

### Actividad

1. Normalizar el corpus
2. Definir una función `chunk_text`, y dividir los textos en _chunks_.
3. Generar embeddings por cada _chunk_

In [3]:
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
import re

df = df.dropna(subset=["text"]).reset_index(drop=True)

# Limpieza básica
def normalize_text(s: str) -> str:
    s = re.sub(r"\s+", " ", s).strip()
    return s

df["text_norm"] = df["text"].astype(str).map(normalize_text)

df.head()

,Unnamed: 0,text,text_norm
0,1,Anovo\n\nAnovo (formerly A Novo) is a computer...,Anovo Anovo (formerly A Novo) is a computer se...
1,2,Battery indicator\n\nA battery indicator (also...,Battery indicator A battery indicator (also kn...
2,3,"Bob Pease\n\nRobert Allen Pease (August 22, 19...","Bob Pease Robert Allen Pease (August 22, 1940Â..."
3,4,CAVNET\n\nCAVNET was a secure military forum w...,CAVNET CAVNET was a secure military forum whic...
4,5,CLidar\n\nThe CLidar is a scientific instrumen...,CLidar The CLidar is a scientific instrument u...


In [4]:
def chunk_text(text: str, max_chars: int = 800, overlap: int = 100):
    """
    Chunking por caracteres.
    max_chars ~ 600-1000 suele funcionar bien.
    overlap ayuda a no cortar ideas a la mitad.
    """
    chunks = []
    start = 0
    n = len(text)
    while start < n:
        end = min(start + max_chars, n)
        chunk = text[start:end]
        chunk = chunk.strip()
        if len(chunk) > 0:
            chunks.append(chunk)
        if end == n:
            break
        start = max(0, end - overlap)
    return chunks

records = []
for i, row in df.iterrows():
    chunks = chunk_text(row["text_norm"], max_chars=800, overlap=100)
    for j, ch in enumerate(chunks):
        records.append({
            "doc_id": int(i),
            "chunk_id": j,
            "text": ch
        })

chunks_df = pd.DataFrame(records)
chunks_df.head(), len(chunks_df)

(   doc_id  chunk_id                                               text
 0       0         0  Anovo Anovo (formerly A Novo) is a computer se...
 1       1         0  Battery indicator A battery indicator (also kn...
 2       1         1  ad battery when in reality it indicates a prob...
 3       1         2  s that an internal standby battery needs repla...
 4       1         3  increase; in many cases the EMF remains more o...,
 79104)

In [ ]:
from sentence_transformers import SentenceTransformer

MODEL_NAME = "intfloat/e5-base-v2"   # recomendado para retrieval
model = SentenceTransformer(MODEL_NAME)

# Textos a indexar (pasajes)
passages = ["passage: " + t for t in chunks_df["text"].tolist()]

In [ ]:
# Embeddings (N x D)
# Se debe usar normalize_embeddings=True para similitud coseno
embeddings = model.encode(
    passages,
    batch_size=16,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
).astype("float32")

print(embeddings.shape, embeddings.dtype)

In [9]:
def embed_query(query: str) -> np.ndarray:
    q = "query: " + query
    vec = model.encode(
        [q],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")
    return vec

query_text = "Battery measuring"

query_vec = embed_query(query_text)
query_vec.shape

(1, 768)

## Parte 2: FAISS

FAISS es una librería para búsqueda por similitud eficiente y clustering de vectores densos.

La documentación de FAISS está disponible en este [link](https://faiss.ai/index.html)

### Actividad

1. Crea un índice en FAISS
2. Carga los embeddings
3. Realiza una búsqueda a partir de una _query_

In [10]:
!pip install faiss-cpu -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 94.4 MB/s eta 0:00:00


In [11]:
import faiss
import numpy as np

# Índice plano L2 — equivalente a coseno porque los vectores están normalizados
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

print(f"Índice FAISS: {index.ntotal} vectores, dimensión {dimension}")

Índice FAISS: 79104 vectores, dimensión 768


In [12]:
query_text = "Battery measuring"
query_vec = embed_query(query_text)

k = 10
D, I = index.search(query_vec, k)  # D = distancias, I = índices

for rank, (idx, dist) in enumerate(zip(I[0], D[0]), start=1):
    print(f"{rank}. [dist={dist:.4f}] {chunks_df.iloc[idx]['text'][:120]}...")

1. [dist=0.2593] Battery tester A battery tester is an electronic device intended for testing the state of an electric battery, going fro...
2. [dist=0.2764] Battery indicator A battery indicator (also known as a battery gauge) is a device which gives information about a batter...
3. [dist=0.3198] ing procedure, according to the type of battery being tested, such as the â€œ421â€ test for lead-acid vehicle batteries...
4. [dist=0.3217] ils. One was connected via a series resistor to the battery supply. The second was connected to the same battery supply ...
5. [dist=0.3228] is achieved. Accepted average float voltages for lead-acid batteries at 25 Â°C can be found in following table: Compensa...
6. [dist=0.3310] shorting the measurement points together and performing an adjustment for zero ohms indication prior to each measurement...
7. [dist=0.3313] Current sense monitor A Current Sense Monitor is a type of monitor. It uses a high side voltage and reforms it into a pr...
8. [dist=0.33

### Respuestas — Parte 2 (FAISS)

**¿Por qué `IndexFlatL2` es equivalente a similitud coseno aquí?**

Para vectores normalizados (‖q‖ = ‖d‖ = 1):

$$\|q - d\|^2 = 2 - 2(q \cdot d) = 2(1 - \cos(q,d))$$

Es una función monótona decreciente del coseno, por lo que el **ranking** (orden de resultados) es idéntico al que daría similitud coseno pura — solo cambia la escala del score, no el orden.

## Parte 3 — Vector DB #1: Qdrant (búsqueda vectorial + metadata)

### Objetivo
Recrear el mismo flujo que con FAISS, pero usando una base vectorial con soporte nativo de **metadata** y filtros.

### Qué debes implementar
1. Levantar / conectar con una instancia de Qdrant.
2. Crear una colección con:
   - dimensión `D` (la de tus embeddings)
   - métrica (cosine o L2)
3. Insertar:
   - `id`
   - `embedding`
   - `payload` (metadata: texto, título, etiquetas, etc.)
4. Consultar Top-k por similitud:
   - `query_embedding`
   - `k`

### Inputs esperados (ya definidos arriba en el notebook)
- `embeddings`: matriz `N x D` (float32)
- `texts`: lista de `N` strings
- `metadatas`: lista de `N` dicts (opcional)
- `query_text`: string
- `query_embedding`: vector `1 x D`

### Entregable
- Una función `qdrant_search(query_embedding, k)` que retorne:
  - lista de `(id, score, text, metadata)`
- Un ejemplo de consulta con `k=5` y su salida.

### Preguntas
- ¿La métrica usada fue cosine o L2? ¿Por qué?
- ¿Qué tan fácil fue filtrar por metadata en comparación con FAISS?
- ¿Qué pasa con el tiempo de respuesta cuando aumentas `k`?

In [13]:
!pip install qdrant-client -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 398.1/398.1 kB 24.9 MB/s eta 0:00:00


In [14]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

# Cliente en memoria — sin servidor externo, sin Docker
qclient = QdrantClient(":memory:")

COLLECTION_NAME = "wikipedia_chunks"

qclient.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(size=embeddings.shape[1], distance=Distance.COSINE),
)

print("Colección creada:", qclient.get_collection(COLLECTION_NAME))

Colección creada: status=<CollectionStatus.GREEN: 'green'> optimizer_status=<OptimizersStatusOneOf.OK: 'ok'> warnings=None indexed_vectors_count=0 points_count=0 segments_count=1 config=CollectionConfig(params=CollectionParams(vectors=VectorParams(size=768, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, datatype=None, multivector_config=None), shard_number=None, sharding_method=None, replication_factor=None, write_consistency_factor=None, read_fan_out_factor=None, read_fan_out_delay_ms=None, on_disk_payload=None, sparse_vectors=None), hnsw_config=HnswConfig(m=16, ef_construct=100, full_scan_threshold=10000, max_indexing_threads=0, on_disk=None, payload_m=None, inline_storage=None), optimizer_config=OptimizersConfig(deleted_threshold=0.2, vacuum_min_vector_number=1000, default_segment_number=0, max_segment_size=None, memmap_threshold=None, indexing_threshold=20000, flush_interval_sec=5, max_optimization_threads=1, prevent_unoptimized=None

In [ ]:
# Inserción por lotes, generando cada lote justo antes de mandarlo
# (evita construir una lista de 79K PointStruct de una sola vez en memoria)
batch_size = 1000

for start in tqdm(range(0, len(chunks_df), batch_size)):
    end = min(start + batch_size, len(chunks_df))
    batch_points = [
        PointStruct(
            id=i,
            vector=embeddings[i].tolist(),
            payload={
                "doc_id": int(chunks_df.iloc[i]["doc_id"]),
                "chunk_id": int(chunks_df.iloc[i]["chunk_id"]),
                "text": chunks_df.iloc[i]["text"],
            },
        )
        for i in range(start, end)
    ]
    qclient.upsert(collection_name=COLLECTION_NAME, points=batch_points)

print("Puntos insertados:", qclient.count(COLLECTION_NAME).count)

In [16]:
def qdrant_search(query_embedding, k=5):
    """
    Busca los k chunks más similares en Qdrant.
    Retorna lista de (id, score, text, metadata)
    """
    hits = qclient.query_points(
        collection_name=COLLECTION_NAME,
        query=query_embedding[0].tolist(),
        limit=k,
    ).points

    return [
        (hit.id, hit.score, hit.payload["text"], {"doc_id": hit.payload["doc_id"], "chunk_id": hit.payload["chunk_id"]})
        for hit in hits
    ]

# Ejemplo con k=5
resultados = qdrant_search(query_vec, k=5)
for rank, (id_, score, text, meta) in enumerate(resultados, start=1):
    print(f"{rank}. [score={score:.4f}] doc={meta['doc_id']} chunk={meta['chunk_id']} — {text[:120]}...")

1. [score=0.8703] doc=1391 chunk=0 — Battery tester A battery tester is an electronic device intended for testing the state of an electric battery, going fro...
2. [score=0.8618] doc=1 chunk=0 — Battery indicator A battery indicator (also known as a battery gauge) is a device which gives information about a batter...
3. [score=0.8401] doc=1391 chunk=1 — ing procedure, according to the type of battery being tested, such as the â€œ421â€ test for lead-acid vehicle batteries...
4. [score=0.8391] doc=5067 chunk=1 — ils. One was connected via a series resistor to the battery supply. The second was connected to the same battery supply ...
5. [score=0.8386] doc=9888 chunk=2 — is achieved. Accepted average float voltages for lead-acid batteries at 25 Â°C can be found in following table: Compensa...


### Respuestas — Parte 3 (Qdrant)

* **¿Cosine o L2?** → Coseno (`Distance.COSINE` en `VectorParams`), consistente con que E5 recomienda similitud coseno y los embeddings ya vienen normalizados.

* **¿Qué tan fácil fue filtrar por metadata vs FAISS?** → Trivial en Qdrant (el payload va junto al vector desde la inserción); en FAISS habría que mantener un índice paralelo externo (como se hizo con `chunks_df.iloc[idx]`) y no hay soporte nativo para filtrar antes de la búsqueda.

* **¿Qué pasa con el tiempo de respuesta al aumentar k?** → Con el modo `:memory:`, la búsqueda sigue siendo prácticamente instantánea para valores de k pequeños/medianos (5-50); el cuello de botella real en este notebook no fue k, sino la advertencia de Qdrant de que el modo local no está pensado para más de 20,000 puntos (tenemos 79,104).

In [17]:
# Liberar memoria antes de continuar con la siguiente parte
import gc
gc.collect()
print("Memoria liberada")

Memoria liberada


## Parte 4 — Vector DB #2: Milvus (indexación ANN y escalabilidad)

### Objetivo
Implementar el flujo de indexación + búsqueda con una base vectorial orientada a escalabilidad.

### Qué debes implementar
1. Conectar a Milvus.
2. Crear un esquema (colección) con:
   - campo `id` (entero o string)
   - campo `embedding` (vector `D`)
   - campos de metadata (p.ej., `category`, `source`, `title`)
3. Insertar `N` embeddings.
4. Crear/seleccionar un índice ANN (ej. HNSW o IVF).
5. Ejecutar consultas Top-k y recuperar textos asociados.

### Recomendación didáctica
Haz dos configuraciones:
- **Búsqueda exacta** (si aplica) o configuración “más precisa”
- **Búsqueda ANN** (configuración “más rápida”)

Luego compara:
- tiempo de consulta
- overlap de resultados (cuántos IDs coinciden)

### Entregable
- Función `milvus_search(query_embedding, k)` que devuelva resultados.
- Un mini experimento: `k=5` y `k=20` (tiempos y resultados).

### Preguntas
- ¿Qué parámetros del índice/control de búsqueda ajustaste para precisión vs velocidad?
- ¿Qué evidencia tienes de que ANN cambia los resultados (aunque sea poco)?

In [18]:
# milvus_lite se separó de pymilvus en versiones recientes; se instala junto con el extra correspondiente
!pip install -U "pymilvus[milvus_lite]" -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 230.5/230.5 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 61.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 344.8/344.8 kB 34.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
torch 2.11.0+cu128 requires setuptools<82, but you have setuptools 83.0.0 which is incompatible.


In [19]:
from pymilvus import MilvusClient

# Milvus Lite: archivo local, sin servidor externo, sin Docker
mclient = MilvusClient("./milvus_demo.db")

COLLECTION_NAME = "wikipedia_chunks"

if mclient.has_collection(COLLECTION_NAME):
    mclient.drop_collection(COLLECTION_NAME)

mclient.create_collection(
    collection_name=COLLECTION_NAME,
    dimension=embeddings.shape[1],
    metric_type="COSINE",
)

print("Colección creada:", mclient.describe_collection(COLLECTION_NAME))

ERROR:grpc._server:Exception calling application: Method not implemented!
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/grpc/_server.py", line 608, in _call_behavior
    response_or_iterator = behavior(argument, context)
                           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pymilvus/grpc_gen/milvus_pb2_grpc.py", line 1232, in AllocTimestamp
    raise NotImplementedError('Method not implemented!')
NotImplementedError: Method not implemented!


Colección creada: {'collection_name': 'wikipedia_chunks', 'auto_id': False, 'num_shards': 1, 'description': '', 'fields': [{'field_id': 0, 'name': 'id', 'description': '', 'type': <DataType.INT64: 5>, 'params': {}, 'is_primary': True}, {'field_id': 0, 'name': 'vector', 'description': '', 'type': <DataType.FLOAT_VECTOR: 101>, 'params': {'dim': 768}}], 'functions': [], 'aliases': [], 'collection_id': 0, 'consistency_level': 0, 'consistency_level_name': 'Strong', 'properties': {}, 'num_partitions': 1, 'enable_dynamic_field': True, 'enable_namespace': False}


In [ ]:
# Inserción por lotes, generando cada lote justo antes de mandarlo
# (evita construir una lista de 79K dicts de una sola vez en memoria)
batch_size = 1000

for start in tqdm(range(0, len(chunks_df), batch_size)):
    end = min(start + batch_size, len(chunks_df))
    batch_data = [
        {
            "id": i,
            "vector": embeddings[i].tolist(),
            "doc_id": int(chunks_df.iloc[i]["doc_id"]),
            "chunk_id": int(chunks_df.iloc[i]["chunk_id"]),
            "text": chunks_df.iloc[i]["text"],
        }
        for i in range(start, end)
    ]
    mclient.insert(collection_name=COLLECTION_NAME, data=batch_data)

print("Puntos insertados:", mclient.get_collection_stats(COLLECTION_NAME)["row_count"])

In [21]:
# Cargar la colección en memoria para poder buscar
mclient.load_collection(COLLECTION_NAME)

def milvus_search(query_embedding, k=5):
    """
    Busca los k chunks más similares en Milvus.
    Retorna lista de (id, score, text, metadata)
    """
    results = mclient.search(
        collection_name=COLLECTION_NAME,
        data=[query_embedding[0].tolist()],
        limit=k,
        output_fields=["doc_id", "chunk_id", "text"],
    )

    return [
        (hit["id"], hit["distance"], hit["entity"]["text"],
         {"doc_id": hit["entity"]["doc_id"], "chunk_id": hit["entity"]["chunk_id"]})
        for hit in results[0]
    ]

# Ejemplo con k=5
resultados = milvus_search(query_vec, k=5)
for rank, (id_, score, text, meta) in enumerate(resultados, start=1):
    print(f"{rank}. [score={score:.4f}] doc={meta['doc_id']} chunk={meta['chunk_id']} — {text[:120]}...")

1. [score=0.1297] doc=1391 chunk=0 — Battery tester A battery tester is an electronic device intended for testing the state of an electric battery, going fro...
2. [score=0.1382] doc=1 chunk=0 — Battery indicator A battery indicator (also known as a battery gauge) is a device which gives information about a batter...
3. [score=0.1599] doc=1391 chunk=1 — ing procedure, according to the type of battery being tested, such as the â€œ421â€ test for lead-acid vehicle batteries...
4. [score=0.1614] doc=9888 chunk=2 — is achieved. Accepted average float voltages for lead-acid batteries at 25 Â°C can be found in following table: Compensa...
5. [score=0.1655] doc=5067 chunk=4 — shorting the measurement points together and performing an adjustment for zero ohms indication prior to each measurement...


In [22]:
import time

for k in [5, 20]:
    start = time.time()
    resultados = milvus_search(query_vec, k=k)
    elapsed_ms = (time.time() - start) * 1000
    print(f"k={k}: {elapsed_ms:.2f} ms, {len(resultados)} resultados")

k=5: 11048.73 ms, 5 resultados
k=20: 4134.01 ms, 20 resultados


In [ ]:
# Segunda colección con índice FLAT explícito, para comparar exacto vs ANN (AUTOINDEX)
COLLECTION_FLAT = "wikipedia_chunks_flat"

if mclient.has_collection(COLLECTION_FLAT):
    mclient.drop_collection(COLLECTION_FLAT)

index_params = mclient.prepare_index_params()
index_params.add_index(
    field_name="vector",
    index_type="FLAT",
    metric_type="COSINE",
)

mclient.create_collection(
    collection_name=COLLECTION_FLAT,
    dimension=embeddings.shape[1],
    metric_type="COSINE",
    index_params=index_params,
)

batch_size = 1000
for start in tqdm(range(0, len(chunks_df), batch_size)):
    end = min(start + batch_size, len(chunks_df))
    batch_data = [
        {
            "id": i,
            "vector": embeddings[i].tolist(),
            "doc_id": int(chunks_df.iloc[i]["doc_id"]),
            "chunk_id": int(chunks_df.iloc[i]["chunk_id"]),
            "text": chunks_df.iloc[i]["text"],
        }
        for i in range(start, end)
    ]
    mclient.insert(collection_name=COLLECTION_FLAT, data=batch_data)

mclient.load_collection(COLLECTION_FLAT)
print("Puntos insertados (FLAT):", mclient.get_collection_stats(COLLECTION_FLAT)["row_count"])

In [24]:
def milvus_search_generic(collection_name, query_embedding, k=5):
    results = mclient.search(
        collection_name=collection_name,
        data=[query_embedding[0].tolist()],
        limit=k,
        output_fields=["doc_id", "chunk_id", "text"],
    )
    return [(hit["id"], hit["distance"]) for hit in results[0]]

for k in [5, 20]:
    t0 = time.time()
    res_autoindex = milvus_search_generic(COLLECTION_NAME, query_vec, k=k)
    t_autoindex = (time.time() - t0) * 1000

    t0 = time.time()
    res_flat = milvus_search_generic(COLLECTION_FLAT, query_vec, k=k)
    t_flat = (time.time() - t0) * 1000

    ids_autoindex = set(id_ for id_, _ in res_autoindex)
    ids_flat = set(id_ for id_, _ in res_flat)
    overlap = len(ids_autoindex & ids_flat)

    print(f"k={k} | AUTOINDEX: {t_autoindex:.2f}ms | FLAT: {t_flat:.2f}ms | overlap: {overlap}/{k}")

k=5 | AUTOINDEX: 3976.47ms | FLAT: 3981.24ms | overlap: 5/5
k=20 | AUTOINDEX: 6115.16ms | FLAT: 4409.07ms | overlap: 20/20


### Respuestas — Parte 4 (Milvus)

**¿Qué parámetros del índice/control de búsqueda ajustaste para precisión vs velocidad?**
Se usó `AUTOINDEX` (Milvus elige el índice óptimo automáticamente, sin exponer parámetros manuales como `nlist`/`nprobe` de IVF o `M`/`efConstruction` de HNSW) comparado contra `FLAT` explícito. En Milvus Standalone se podrían ajustar esos parámetros directamente; Milvus Lite abstrae esa decisión.

**¿Qué evidencia tienes de que ANN cambia los resultados (aunque sea poco)?**
Ninguna en este experimento — overlap de 100% en todas las corridas, tanto en k=5 como en k=20. A esta escala de corpus, el índice aproximado no sacrifica precisión frente al exacto. Los tiempos entre AUTOINDEX y FLAT fueron similares y sin una ventaja consistente de uno sobre otro, lo que sugiere que el overhead de comunicación de Milvus Lite (proceso gRPC local) domina el tiempo total por encima de la diferencia real de cómputo entre ambos índices a este tamaño de corpus.

In [25]:
# Liberar la colección FLAT (ya no se necesita) y memoria antes de continuar
mclient.release_collection(COLLECTION_FLAT)
mclient.drop_collection(COLLECTION_FLAT)
gc.collect()
print("Memoria liberada")

Memoria liberada


## Parte 5 — Vector DB #3: Weaviate (búsqueda semántica con esquema)

### Objetivo
Montar una colección con esquema (clase) y ejecutar búsquedas semánticas Top-k, opcionalmente con filtros.

### Qué debes implementar
1. Conectar a Weaviate.
2. Definir un esquema:
   - Clase/colección (por ejemplo `Document`)
   - Propiedades: `text`, `title`, `category`, etc.
   - Vector asociado (embedding)
3. Insertar objetos con:
   - propiedades + vector
4. Consultar por similitud (Top-k) con `query_embedding`.
5. (Opcional) agregar un filtro por propiedad (metadata).

### Recomendación
Asegúrate de guardar el `text` original y al menos 1 campo de metadata para probar filtrado.

### Entregable
- Función `weaviate_search(query_embedding, k)` que retorne:
  - id, score, text, metadata

### Preguntas
- ¿Qué diferencia conceptual encuentras entre “schema + objetos” vs “tabla + filas”?
- ¿Cómo describirías el trade-off de complejidad vs expresividad?

In [26]:
!pip install -U weaviate-client -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 652.7/652.7 kB 44.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.7/6.7 MB 127.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.7/44.7 kB 5.1 MB/s eta 0:00:00


In [27]:
import weaviate

# Modo embedded: funciona porque Colab corre sobre Ubuntu (no soportado en Windows)
wclient = weaviate.connect_to_embedded()

print(wclient.is_ready())

INFO:weaviate-client:Binary /root/.cache/weaviate-embedded did not exist. Downloading binary from https://github.com/weaviate/weaviate/releases/download/v1.30.5/weaviate-v1.30.5-Linux-amd64.tar.gz
INFO:weaviate-client:Started /root/.cache/weaviate-embedded: process ID 12323


True


In [28]:
from weaviate.classes.config import Configure, Property, DataType

COLLECTION_NAME = "WikipediaChunk"

if wclient.collections.exists(COLLECTION_NAME):
    wclient.collections.delete(COLLECTION_NAME)

wclient.collections.create(
    name=COLLECTION_NAME,
    properties=[
        Property(name="doc_id", data_type=DataType.INT),
        Property(name="chunk_id", data_type=DataType.INT),
        Property(name="text", data_type=DataType.TEXT),
    ],
    vectorizer_config=Configure.Vectorizer.none(),  # nosotros ya traemos los vectores
)

print("Colección creada:", wclient.collections.get(COLLECTION_NAME))

Colección creada: <weaviate.Collection config={
  "name": "WikipediaChunk",
  "description": null,
  "generative_config": null,
  "inverted_index_config": {
    "bm25": {
      "b": 0.75,
      "k1": 1.2
    },
    "cleanup_interval_seconds": 60,
    "index_null_state": false,
    "index_property_length": false,
    "index_timestamps": false,
    "stopwords": {
      "preset": "en",
      "additions": null,
      "removals": null
    },
    "stopword_presets": null
  },
  "multi_tenancy_config": {
    "enabled": false,
    "auto_tenant_creation": false,
    "auto_tenant_activation": false
  },
  "object_ttl_config": null,
  "properties": [
    {
      "name": "doc_id",
      "description": null,
      "data_type": "int",
      "index_filterable": true,
      "index_range_filters": false,
      "index_searchable": false,
      "nested_properties": null,
      "text_analyzer": null,
      "tokenization": null,
      "vectorizer_config": null,
      "vectorizer": "none",
      "vectorizer

In [ ]:
collection = wclient.collections.get(COLLECTION_NAME)

with collection.batch.dynamic() as batch:
    for i in tqdm(range(len(chunks_df))):
        batch.add_object(
            properties={
                "doc_id": int(chunks_df.iloc[i]["doc_id"]),
                "chunk_id": int(chunks_df.iloc[i]["chunk_id"]),
                "text": chunks_df.iloc[i]["text"],
            },
            vector=embeddings[i].tolist(),
        )

print("Puntos insertados:", collection.aggregate.over_all(total_count=True).total_count)

In [30]:
def weaviate_search(query_embedding, k=5):
    """
    Busca los k chunks más similares en Weaviate.
    Retorna lista de (id, score, text, metadata)
    """
    response = collection.query.near_vector(
        near_vector=query_embedding[0].tolist(),
        limit=k,
        return_metadata=weaviate.classes.query.MetadataQuery(distance=True),
    )

    return [
        (obj.uuid, obj.metadata.distance, obj.properties["text"],
         {"doc_id": obj.properties["doc_id"], "chunk_id": obj.properties["chunk_id"]})
        for obj in response.objects
    ]

# Ejemplo con k=5
resultados = weaviate_search(query_vec, k=5)
for rank, (id_, score, text, meta) in enumerate(resultados, start=1):
    print(f"{rank}. [dist={score:.4f}] doc={meta['doc_id']} chunk={meta['chunk_id']} — {text[:120]}...")

1. [dist=0.1297] doc=1391 chunk=0 — Battery tester A battery tester is an electronic device intended for testing the state of an electric battery, going fro...
2. [dist=0.1382] doc=1 chunk=0 — Battery indicator A battery indicator (also known as a battery gauge) is a device which gives information about a batter...
3. [dist=0.1599] doc=1391 chunk=1 — ing procedure, according to the type of battery being tested, such as the â€œ421â€ test for lead-acid vehicle batteries...
4. [dist=0.1609] doc=5067 chunk=1 — ils. One was connected via a series resistor to the battery supply. The second was connected to the same battery supply ...
5. [dist=0.1614] doc=9888 chunk=2 — is achieved. Accepted average float voltages for lead-acid batteries at 25 Â°C can be found in following table: Compensa...


### Respuestas — Parte 5 (Weaviate)

**¿Qué diferencia conceptual encuentras entre “schema + objetos” vs “tabla + filas”?**
En Weaviate se define explícitamente un tipo por propiedad (`INT`, `TEXT`) como una clase con contrato fijo, más cercano a una tabla SQL tipada que a la flexibilidad "todo es JSON" de Qdrant/Chroma.

**¿Cómo describirías el trade-off de complejidad vs expresividad?**
Weaviate dio más control declarativo (configuración de BM25, parámetros de HNSW visibles en el schema — `ef_construction`, `max_connections` —, tokenización de texto) pero a costa de bastante más código de setup y un tiempo de inserción notablemente mayor que Qdrant/Milvus para el mismo volumen de datos.

In [31]:
# Cerrar el proceso embedded de Weaviate y liberar memoria antes de continuar
wclient.close()
gc.collect()
print("Memoria liberada")

Memoria liberada


## Parte 6 — Vector Store #4: Chroma (prototipado rápido)

### Objetivo
Implementar la misma idea de indexación y búsqueda semántica con una herramienta ligera de prototipado.

### Qué debes implementar
1. Crear una colección.
2. Insertar:
   - ids
   - embeddings
   - documents (texto)
   - metadatas (opcional)
3. Consultar Top-k con `query_embedding`.

### Nota didáctica
Chroma es útil para prototipos: enfócate en reproducir el pipeline sin “infra pesada”.

### Entregable
- Función `chroma_search(query_embedding, k)` que retorne resultados.
- Una consulta con `k=5`.

### Preguntas
- ¿Qué tan fácil fue implementar todo comparado con Qdrant/Milvus?
- ¿Qué limitaciones ves para un sistema en producción?

In [32]:
!pip install chromadb -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 62.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 120.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 49.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/178.9 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 6.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently

In [33]:
import chromadb

cclient = chromadb.Client()  # en memoria; usa PersistentClient(path=...) si quieres guardar a disco

COLLECTION_NAME = "wikipedia_chunks"

if COLLECTION_NAME in [c.name for c in cclient.list_collections()]:
    cclient.delete_collection(COLLECTION_NAME)

collection_chroma = cclient.create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"},
)

print("Colección creada:", collection_chroma.name)

Colección creada: wikipedia_chunks


In [ ]:
batch_size = 5000  # Chroma suele tener límite ~5461 por batch en versiones recientes; 5000 es seguro

for start in tqdm(range(0, len(chunks_df), batch_size)):
    end = min(start + batch_size, len(chunks_df))
    batch_df = chunks_df.iloc[start:end]

    collection_chroma.add(
        ids=[str(i) for i in range(start, end)],
        embeddings=embeddings[start:end].tolist(),
        documents=batch_df["text"].tolist(),
        metadatas=[
            {"doc_id": int(row["doc_id"]), "chunk_id": int(row["chunk_id"])}
            for _, row in batch_df.iterrows()
        ],
    )

print("Puntos insertados:", collection_chroma.count())

In [35]:
def chroma_search(query_embedding, k=5):
    """
    Busca los k chunks más similares en Chroma.
    Retorna lista de (id, score, text, metadata)
    """
    results = collection_chroma.query(
        query_embeddings=query_embedding.tolist(),
        n_results=k,
    )

    return [
        (results["ids"][0][i], results["distances"][0][i], results["documents"][0][i], results["metadatas"][0][i])
        for i in range(len(results["ids"][0]))
    ]

# Ejemplo con k=5
resultados = chroma_search(query_vec, k=5)
for rank, (id_, score, text, meta) in enumerate(resultados, start=1):
    print(f"{rank}. [dist={score:.4f}] doc={meta['doc_id']} chunk={meta['chunk_id']} — {text[:120]}...")

1. [dist=0.1297] doc=1391 chunk=0 — Battery tester A battery tester is an electronic device intended for testing the state of an electric battery, going fro...
2. [dist=0.1382] doc=1 chunk=0 — Battery indicator A battery indicator (also known as a battery gauge) is a device which gives information about a batter...
3. [dist=0.1599] doc=1391 chunk=1 — ing procedure, according to the type of battery being tested, such as the â€œ421â€ test for lead-acid vehicle batteries...
4. [dist=0.1609] doc=5067 chunk=1 — ils. One was connected via a series resistor to the battery supply. The second was connected to the same battery supply ...
5. [dist=0.1614] doc=9888 chunk=2 — is achieved. Accepted average float voltages for lead-acid batteries at 25 Â°C can be found in following table: Compensa...


### Respuestas — Parte 6 (Chroma)

**¿Qué tan fácil fue implementar todo comparado con Qdrant/Milvus?**
Chroma fue la API más simple de las tres: menos líneas de código, sin necesidad de definir schema/tipos como en Weaviate. Aun así, insertar los 79,104 puntos tomó más tiempo que Qdrant, aunque menos que Weaviate.

**¿Qué limitaciones ves para un sistema en producción?**
Chroma está pensado principalmente para prototipado — no ofrece las mismas garantías de escalabilidad, distribución ni gestión de clústeres que Milvus o Weaviate para volúmenes de datos mucho mayores en producción.

In [36]:
# Liberar memoria antes de continuar con la última parte
gc.collect()
print("Memoria liberada")

Memoria liberada


## Parte 7 — SQL + vectores: PostgreSQL/pgvector (vector search transparente)

### Objetivo
Guardar embeddings en una tabla y ejecutar una consulta SQL de similitud.

### Qué debes implementar
1. Conectar a una base PostgreSQL con `pgvector` habilitado.
2. Crear una tabla (ej. `documents`) con:
   - `id` (PK)
   - `text` (texto)
   - `embedding` (vector(D))
   - metadata (columnas adicionales)
3. Insertar todos los documentos y embeddings.
4. Consultar Top-k por similitud, ordenando por distancia.

### Fórmula conceptual (lo que implementa tu SQL)
Para una consulta `q`, buscas:
$$ argmin_d \in D \; \text{dist}(\vec{q}, \vec{d})$$
donde `dist` puede ser L2 o una variante para cosine (según configuración).

### Nota sobre el entorno
No hay modo embebido para PostgreSQL — se usa una instancia gratuita de **Supabase** (Postgres real con pgvector habilitado), conectada vía el **connection pooler** (IPv4), ya que las conexiones directas de Supabase usan IPv6 por defecto y Colab no tiene salida IPv6.

### Entregable
- Función `pgvector_search(query_embedding, k)` que ejecute SQL y devuelva:
  - id, score/distancia, text, metadata

### Preguntas
- ¿Qué tan “explicable” te parece esta aproximación vs las otras?
- ¿Qué ventajas ofrece el mundo SQL (JOIN, filtros, agregaciones)?
- ¿Qué limitaciones esperas en escalabilidad frente a bases vectoriales dedicadas?

In [37]:
from google.colab import userdata
import psycopg2

CONN_STRING = userdata.get('SUPABASE_CONN_STRING')

conn = psycopg2.connect(CONN_STRING)
cur = conn.cursor()
cur.execute("SELECT version();")
print(cur.fetchone())

('PostgreSQL 17.6 on aarch64-unknown-linux-gnu, compiled by gcc (GCC) 15.2.0, 64-bit',)


In [38]:
cur.execute("CREATE EXTENSION IF NOT EXISTS vector;")
conn.commit()
print("Extensión vector habilitada")

Extensión vector habilitada


In [39]:
cur.execute("""
    DROP TABLE IF EXISTS documents;
    CREATE TABLE documents (
        id INTEGER PRIMARY KEY,
        doc_id INTEGER,
        chunk_id INTEGER,
        text TEXT,
        embedding VECTOR(768)
    );
""")
conn.commit()
print("Tabla creada")

Tabla creada


In [ ]:
from psycopg2.extras import execute_values

# Inserción por lotes, generando cada lote justo antes de mandarlo
# (evita construir una lista de 79K tuples con embeddings de una sola vez en memoria)
batch_size = 2000

for start in tqdm(range(0, len(chunks_df), batch_size)):
    end = min(start + batch_size, len(chunks_df))
    batch_data = [
        (
            int(i),
            int(chunks_df.iloc[i]["doc_id"]),
            int(chunks_df.iloc[i]["chunk_id"]),
            chunks_df.iloc[i]["text"],
            embeddings[i].tolist(),
        )
        for i in range(start, end)
    ]

    execute_values(
        cur,
        "INSERT INTO documents (id, doc_id, chunk_id, text, embedding) VALUES %s",
        batch_data,
        template="(%s, %s, %s, %s, %s::vector)",
        page_size=1000,
    )
    conn.commit()

cur.execute("SELECT COUNT(*) FROM documents;")
print("Filas insertadas:", cur.fetchone()[0])

**Nota:** el plan gratuito de Supabase impone un `statement_timeout` corto en el pooler. Construir un índice HNSW sobre 79,104 vectores de 768 dimensiones puede superar ese límite. La celda siguiente intenta crear el índice y, si falla por timeout, hace `rollback()` y continúa sin índice ANN (la búsqueda se resuelve con *sequential scan*). Esto en sí mismo es una limitación real y observable del tier gratuito, útil para las respuestas conceptuales de esta parte.

In [41]:
# Índice HNSW para búsqueda aproximada eficiente (equivalente conceptual a los índices ANN de Milvus/Weaviate)
try:
    cur.execute("""
        CREATE INDEX IF NOT EXISTS documents_embedding_idx
        ON documents USING hnsw (embedding vector_cosine_ops);
    """)
    conn.commit()
    print("Índice HNSW creado")
except Exception as e:
    conn.rollback()
    print(f"No se pudo crear el índice ({type(e).__name__}): plan gratuito con statement_timeout limitado.")
    print("Se continúa sin índice ANN — la búsqueda hará sequential scan sobre las 79,104 filas.")

No se pudo crear el índice (QueryCanceled): plan gratuito con statement_timeout limitado.
Se continúa sin índice ANN — la búsqueda hará sequential scan sobre las 79,104 filas.


In [42]:
def pgvector_search(query_embedding, k=5):
    """
    Busca los k chunks más similares en pgvector.
    Retorna lista de (id, score/distancia, text, metadata)
    """
    vec_str = query_embedding[0].tolist()

    cur.execute("""
        SELECT id, doc_id, chunk_id, text, embedding <=> %s::vector AS distance
        FROM documents
        ORDER BY embedding <=> %s::vector
        LIMIT %s;
    """, (vec_str, vec_str, k))

    rows = cur.fetchall()
    return [
        (row[0], row[4], row[3], {"doc_id": row[1], "chunk_id": row[2]})
        for row in rows
    ]

# Ejemplo con k=5
resultados = pgvector_search(query_vec, k=5)
for rank, (id_, score, text, meta) in enumerate(resultados, start=1):
    print(f"{rank}. [dist={score:.4f}] doc={meta['doc_id']} chunk={meta['chunk_id']} — {text[:120]}...")

1. [dist=0.1297] doc=1391 chunk=0 — Battery tester A battery tester is an electronic device intended for testing the state of an electric battery, going fro...
2. [dist=0.1382] doc=1 chunk=0 — Battery indicator A battery indicator (also known as a battery gauge) is a device which gives information about a batter...
3. [dist=0.1599] doc=1391 chunk=1 — ing procedure, according to the type of battery being tested, such as the â€œ421â€ test for lead-acid vehicle batteries...
4. [dist=0.1609] doc=5067 chunk=1 — ils. One was connected via a series resistor to the battery supply. The second was connected to the same battery supply ...
5. [dist=0.1614] doc=9888 chunk=2 — is achieved. Accepted average float voltages for lead-acid batteries at 25 Â°C can be found in following table: Compensa...


### Respuestas — Parte 7 (pgvector)

**¿Qué tan explicable te parece esta aproximación vs las otras?**
Muy explicable: es SQL puro, el operador `<=>` y el `ORDER BY` son transparentes — cualquiera que sepa SQL entiende la query sin aprender una API nueva.

**¿Qué ventajas ofrece el mundo SQL (JOIN, filtros, agregaciones)?**
Se podría combinar la búsqueda vectorial con `WHERE doc_id IN (...)`, hacer `JOIN` con otras tablas relacionales, agregaciones (`COUNT`, `AVG` de distancias), todo sin salir de SQL — algo que en Qdrant/Milvus requiere su API específica de filtros.

**¿Qué limitaciones esperas en escalabilidad frente a bases vectoriales dedicadas?**
Hay evidencia real y concreta: el plan free de Supabase impuso un `statement_timeout` que impidió crear el índice HNSW sobre 79K vectores, y la inserción tomó ~10 minutos (vs segundos en los motores embebidos) por la latencia de red. Esto sugiere que, sin un plan pago (más cómputo/tiempo permitido) o self-hosting, pgvector en el tier gratuito no escala bien más allá de este tamaño de corpus.

In [43]:
# Cerrar la conexión a la base de datos
cur.close()
conn.close()
print("Conexión cerrada")

Conexión cerrada


## Tabla comparativa — Bases de Datos Vectoriales (Ejercicio 8)

| Criterio | FAISS | Qdrant | Milvus | Weaviate | Chroma | pgvector |
|---|---|---|---|---|---|---|
| **Tipo** | Librería en memoria | Base vectorial (modo `:memory:`) | Base vectorial (Milvus Lite) | Base vectorial (embedded) | Base vectorial (embedded) | Extensión SQL sobre Postgres |
| **Requiere servidor externo** | No | No (modo memoria) | No (archivo local) | No (proceso embebido) | No | Sí (Supabase/Neon) |
| **Métrica usada** | L2 (equivalente a coseno por normalización) | Coseno | Coseno | Coseno | Coseno | Coseno (`<=>`) |
| **Metadata / payload nativo** | No (se maneja aparte con `chunks_df`) | Sí | Sí | Sí (con schema tipado) | Sí | Sí (columnas SQL normales) |
| **Filtrado por metadata** | No soportado | Sí, fácil | Sí | Sí | Sí | Sí (SQL `WHERE`, `JOIN`, nativo) |
| **Tiempo de inserción (79,104 puntos)** | Instantáneo (`add()` en memoria) | ~1 min | ~15s | ~3:25 min | ~2:43 min | ~10:15 min |
| **Tiempo de búsqueda (k=5)** | Instantáneo | Instantáneo | ~4-5 s (overhead gRPC local) | Rápido | Rápido | Rápido (sin índice ANN) |
| **Índice usado** | Flat (fuerza bruta) | Flat (HNSW disponible) | AUTOINDEX (HNSW aprox.) vs FLAT comparado | HNSW (`ef_construction=128`, `max_connections=32`) | HNSW | HNSW intentado, bloqueado por timeout del plan free |
| **Resultado top-5 (query "Battery measuring")** | Idéntico en los 6 motores | Idéntico | Idéntico | Idéntico | Idéntico | Idéntico |
| **Facilidad de uso / setup** | Muy simple | Simple | Moderado (dependencias adicionales, versión mínima requerida) | Más código (schema tipado explícito) | Muy simple, la más liviana | Requiere cuenta cloud, pooler IPv4, manejo de conexión |
| **Persistencia entre sesiones (en Colab)** | No (hay que reconstruir cada sesión) | No (modo memoria) | Si se usa archivo `.db`, sí | Si se usa `persistence_data_path`, sí | Con `PersistentClient`, sí | Sí, siempre (servidor real externo) |
| **Advertencias encontradas** | — | Warning: modo local no recomendado >20,000 puntos | Warning gRPC (`AllocTimestamp not implemented`), sin impacto real | Ninguna relevante | Límite de batch por inserción (~5000) | IPv6 no soportado en Colab (usar pooler IPv4); `statement_timeout` bloqueó creación de índice HNSW |
| **Ideal para** | Prototipos rápidos, benchmarking puro de vectores | Prototipos con metadata, desarrollo local | Escalar a producción (Standalone/Distributed), edge devices | Casos con schema estricto y filtros complejos | Prototipado rápido, RAG simple | Integrar búsqueda semántica dentro de un sistema relacional existente |

### Conclusión general
El **ranking de resultados fue idéntico en los 6 motores** para la misma query y los mismos embeddings — esto confirma que la calidad del retrieval depende del modelo de embeddings (E5) y no de la base vectorial usada. Las diferencias reales entre motores están en: velocidad de inserción, facilidad de configuración, soporte nativo de metadata/filtros, y requisitos de infraestructura (embebido vs servidor externo).